# Lane 4: Investment Research Agent

Owner: Ian Schmitt. Development notebook; finished cells are copied into `notebook.ipynb` section 4 at assembly. See `docs/` and `setup.md`.

In [ ]:
import json
import sys
from collections import Counter
sys.path.insert(0, "..")  # this notebook lives in dev/; make src/ importable

from src import llm, tools
from src.llm import chat, handoff


Plans its research, calls tools dynamically, reflects on its output, and learns across runs.

**Contract:** `ResearchAgent.plan(symbol) -> [{"tool", "args", "why"}]` (memory injected into the prompt, max 8 steps); `run(symbol) -> dict` (dispatches by name over `tools.TOOLS` plus `run_chain` and `route_and_analyze`, logs every call, writes the report through Lane 3's loop); `reflect(symbol, run_result) -> {"score", "note", "lesson"}` (takes `run()`'s output, the plan, tool log, report, and score history; scores the report with Lane 3's `evaluate` and writes a symbol-specific note plus an optional general lesson); `remember(symbol, note, lesson=None)` (appends the note under the symbol and the lesson to `lessons` in `self.memory`, a dict that starts empty on every notebook run).

**Demonstration:** AAPL twice, NVDA once. Run 2's plan must visibly change because of the stored notes; run the planner at temperature 0 so memory is the only input that differs.

During development, import the other lanes' functions from their script exports, which live next to this notebook (regenerate with `uv run jupyter nbconvert --to script dev/chain.ipynb`):

```python
from chain import run_chain
from route import route_and_analyze
from evaluate import evaluator_optimizer
```

### Stand-ins (development only)

Until the lanes deliver on Oct 4, these four functions stand in for them with the same names, arguments, and return shapes, so the agent can be built and run now. `run_chain` and `route_and_analyze` make no LLM calls; `evaluate` and `evaluator_optimizer` make real ones so reflection has a real score to work with. When the exports exist, replace this cell with the three imports above. This cell is never copied into `notebook.ipynb`.


In [ ]:
# DEV ONLY. Same names, arguments, and return shapes as the lanes' functions.


# Lane 1 stand-in: the cached news, every article labeled neutral, no entities.
def run_chain(symbol):
    articles = [{**a, "sentiment": "neutral", "topic": "news", "entities": []} for a in tools.get_news(symbol)]
    return {"articles": articles, "summary": f"Stand-in summary of {len(articles)} {symbol} headlines."}


# Lane 2 stand-in: every item routed to news, with a placeholder analysis.
def route_and_analyze(symbol, items):
    return [{"item": item, "route": "news", "analysis": f"Stand-in analysis of: {item['title']}"} for item in items]


# Lane 3 stand-ins: one real Critic call, and at most one real revision.
def evaluate(text, criteria):
    system = (
        "You are the Critic. Check the report against each criterion. Reply as JSON: "
        '{"score": 1-10, "pass": true or false, "feedback": ["<criterion>: <what is missing or wrong>", ...]}. '
        "pass is true only if every criterion is met; feedback is empty when pass is true."
    )
    return chat(system, f"CRITERIA:\n{criteria}\n\nREPORT:\n{text}", json_mode=True, temperature=0, agent="Critic")


def evaluator_optimizer(generate, criteria, max_rounds=2):
    text = generate()
    check = evaluate(text, criteria)
    history = [{"round": 0, **check}]
    if not check["pass"]:
        system = ("You are the Writer. Revise the report to address every note and change nothing else. "
                  "Reply with the revised report only, no commentary on what changed.")
        text = chat(system, f"NOTES:\n{json.dumps(check['feedback'])}\n\nREPORT:\n{text}", agent="Writer")
        check = evaluate(text, criteria)
        history.append({"round": 1, **check})
    return {"final": text, "history": history}


In [ ]:
# Every step the planner can choose, by name. The executor looks the name up here and calls it.
# Built once when this cell runs, so the lane functions must be defined (or imported) above it.
STEPS = {**tools.TOOLS, "run_chain": run_chain, "route_and_analyze": route_and_analyze}

# What the Coordinator reads when planning: what each step returns, what it costs, and its args.
# The args shown are the cached ones; any other value fetches live from Yahoo.
MENU = """\
get_prices  args {"period": "6mo"}  Daily closes plus last_close, change_pct, high, low, avg_volume. No LLM calls.
get_fundamentals  args {}  Name, sector, market cap and revenue (billions), P/E, revenue and earnings growth and
    profit margin (percent), 52-week range. No LLM calls.
get_news  args {"limit": 10}  Recent headlines with summaries, unlabeled. No LLM calls.
run_chain  args {}  Fetches the recent news itself, labels each article's sentiment and topic, extracts
    entities, and writes one summary. One LLM call per article.
route_and_analyze  args {}  Sends each article to an earnings, news, or market specialist for a short analysis.
    Two LLM calls per article. Uses run_chain's labeled articles if run_chain ran earlier in the plan,
    otherwise the raw news.
"""


In [ ]:
# The figures a note would cite, from whichever steps ran, as one JSON string. The Writer, the Critic's
# reference data, and the Reflector all read this view, so the numbers the Writer quotes are the numbers
# the Critic checks. The 125 daily closes are dropped and specialist analyses trimmed to keep prompts small.
def brief(results):
    view = {}
    if "get_prices" in results:
        view["prices"] = {k: v for k, v in results["get_prices"].items() if k not in ("dates", "close")}
    if "get_fundamentals" in results:
        view["fundamentals"] = results["get_fundamentals"]
    if "get_news" in results:
        view["headlines"] = [article["title"] for article in results["get_news"]]
    if "run_chain" in results:
        view["news_summary"] = results["run_chain"]["summary"]
        view["sentiment_counts"] = dict(Counter(a["sentiment"] for a in results["run_chain"]["articles"]))
    if "route_and_analyze" in results:
        view["specialist_analyses"] = [
            {"route": row["route"], "title": row["item"]["title"], "analysis": row["analysis"][:300]}
            for row in results["route_and_analyze"]
        ]
    return json.dumps(view, indent=1)


WRITER = """\
You are the Writer on an investment research team. Write a research note of 150 to 250 words on the
stock below, using only the data given; if the data does not cover something, leave it out.
Write figures in plain words with their units ("revenue of 416.2 billion USD", "down 4.3 percent over
six months"), never as data field names. Quote figures exactly as given and compute no new ones.
Do not recommend buying, selling, or holding.
"""


# The Writer's draft. Passed to Lane 3's loop as a zero-argument generate, so the loop never needs
# to know where the text comes from.
def write_report(symbol, results):
    return chat(WRITER, f"Stock: {symbol}\n\nData:\n{brief(results)}", agent="Writer")


# What the Critic grades every note against. Fixed, whatever the plan gathered: a plan that skipped
# a needed step produces a note that fails a criterion, and that failure is what reflection learns from.
CRITERIA = """\
1. States the share price change over the period, with its percent figure.
2. Cites at least one fundamental (revenue, market cap, P/E, growth, or margin) with its unit.
3. States the sentiment of recent news as counts of positive, negative, and neutral articles.
4. Names at least one specific risk drawn from the data.
5. Makes no recommendation to buy, sell, or hold.
6. Every figure in the note matches the reference data (rounding is fine).
"""


# The criteria plus the data the note was written from, so the Critic can check that a figure is
# correct, not only that one is present.
def report_criteria(results):
    return f"{CRITERIA}\nReference data:\n{brief(results)}"


# Run one plan step by name. route_and_analyze is the one step that needs another step's output: it
# takes run_chain's labeled articles if the chain ran, otherwise the raw news. Everything else takes
# the symbol plus the args from the plan.
def call_step(name, args, symbol, results):
    if name not in STEPS:
        raise ValueError(f"unknown step {name!r}; the menu has {', '.join(STEPS)}")
    if name == "route_and_analyze":
        items = results["run_chain"]["articles"] if "run_chain" in results else tools.get_news(symbol)
        return STEPS[name](symbol, items)
    return STEPS[name](symbol, **args)


In [ ]:
# The Coordinator's instructions. It sees the goal and the menu, never the Critic's criteria:
# what a good note needs is something the agent has to learn from feedback, run by run.
PLANNER = f"""\
You are the Coordinator of an investment research team. Plan the research for one stock symbol.
The team will write a short research note on the stock from the data your plan gathers.
The note states facts and never recommends buying, selling, or holding.

Steps you can run:
{MENU}
Rules:
- Choose only the steps the note needs. LLM calls cost time; do not run a step the note will not use.
- If you run route_and_analyze, run run_chain before it.
- At most 8 steps.
- If a note or lesson from earlier runs leads you to add, drop, or change a step, that step's why says
  which note and what it changed.

Reply as JSON: {{"steps": [{{"tool": "<step name>", "args": {{...}}, "why": "<one sentence>"}}]}}
"""


class ResearchAgent:
    # Memory lives on the agent and starts empty, so every notebook run is a cold start.
    # symbols holds notes per symbol; lessons holds notes that apply to any symbol.
    def __init__(self):
        self.memory = {"symbols": {}, "lessons": []}

    # Ask the Coordinator for an ordered list of steps {"tool", "args", "why"}, with memory in the
    # prompt. Temperature 0, so between two runs on one symbol the only input that differs is memory.
    def plan(self, symbol):
        notes = self.memory["symbols"].get(symbol, [])
        recall = (
            f"Symbol: {symbol}\n\n"
            f"Notes from earlier runs on {symbol}:\n" + ("\n".join(f"- {n}" for n in notes) or "- none yet") + "\n\n"
            "Lessons for any symbol:\n" + ("\n".join(f"- {n}" for n in self.memory["lessons"]) or "- none yet")
        )
        reply = chat(PLANNER, recall, json_mode=True, temperature=0, agent="Coordinator")
        # JSON mode returns an object, so the list arrives wrapped in "steps". Unknown tool names are
        # kept: the executor logs and skips them, so a planning mistake stays visible.
        return [
            {"tool": step.get("tool"), "args": step.get("args") or {}, "why": step.get("why", "")}
            for step in reply.get("steps", [])[:8]
        ]

    # Plan, execute the plan step by step, then write the note through Lane 3's loop. Returns
    # everything reflect() needs. A failed step is logged and skipped, not raised: a run that loses
    # one source should still produce a note, and the failure is evidence for reflection.
    def run(self, symbol):
        first_call = len(llm.CALL_LOG)
        plan = self.plan(symbol)
        results, log = {}, []
        for step in plan:
            handoff("Coordinator", step["tool"], step["why"])
            try:
                output = call_step(step["tool"], step["args"], symbol, results)
            except Exception as error:  # noqa: BLE001  any failure, from bad args to a timeout, is logged
                log.append({**step, "ok": False, "result": f"{type(error).__name__}: {error}"})
                print(f"    failed: {log[-1]['result']}")
                continue
            results[step["tool"]] = output
            log.append({**step, "ok": True, "result": str(output)[:80]})

        criteria = report_criteria(results)
        handoff("Coordinator", "Writer", f"{symbol}: data from {', '.join(results)}")
        loop = evaluator_optimizer(lambda: write_report(symbol, results), criteria)
        handoff("Critic", "Coordinator", f"final score {loop['history'][-1]['score']}/10")
        return {
            "symbol": symbol, "plan": plan, "log": log, "results": results, "criteria": criteria,
            "report": loop["final"], "history": loop["history"], "calls": llm.CALL_LOG[first_call:],
        }

    # Score run()'s report and write {"score", "note", "lesson"}; the note is specific to
    # this symbol, the lesson (or None) applies to any symbol.
    def reflect(self, symbol, run_result):
        raise NotImplementedError

    # Append the note under this symbol and the lesson to the shared lessons list.
    def remember(self, symbol, note, lesson=None):
        raise NotImplementedError


## Demonstration

Run the function on cached data and produce the table and figure described above. Keep printed output narrow enough for a PDF page.

Put demo code inside the `if __name__ == "__main__":` block below. It runs normally in this notebook, but not when another lane imports this notebook's `.py` export, so importing your functions never re-runs your LLM calls and plots.

In [ ]:
if __name__ == "__main__":
    # demo code goes here, indented under this line
    pass
